In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_groq import ChatGroq

In [ ]:
load_dotenv(override=True)


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

key = os.getenv("GROQ_API_KEY")

print("Key exists:", key is not None)
print("Key starts with:", key[:7] if key else None)
print("Key length:", len(key) if key else 0)

In [ ]:


model = ChatGroq(
    model="qwen/qwen3.8-27b",
 
)

In [ ]:
response = model.invoke("Say hello")

print(response.content)

In [ ]:
class PPSC_schema(BaseModel):
    feedback: str = Field(description='detailed feedback for essay ')
    score : int = Field(description='score out of 10',le= 10,ge=0)
    

In [ ]:
structured_model=model.with_structured_output(PPSC_schema)

In [ ]:
essay="""Politics in Pakistan
Introduction

Politics plays an important role in the development of every country. In Pakistan, politics has had a major influence on the country's government, economy, society, and foreign relations. Since its independence in 1947, Pakistan has experienced democratic governments, military rule, political changes, and constitutional developments. Political instability has remained one of the major challenges for the country.

History of Politics in Pakistan

Pakistan became an independent country on 14 August 1947. In the early years, the country faced many difficulties, including the settlement of refugees, economic problems, and the absence of strong political institutions. The early death of Quaid-e-Azam Muhammad Ali Jinnah and the assassination of Liaquat Ali Khan also created political difficulties.

Pakistan adopted its first Constitution in 1956, but it remained in force for only a short period. In 1958, martial law was imposed and political activities were suspended. Later, Pakistan experienced several periods of military government, particularly under General Ayub Khan, General Zia-ul-Haq, and General Pervez Musharraf.

Democratic governments also ruled Pakistan for different periods. Political parties such as the Pakistan Peoples Party (PPP), Pakistan Muslim League (PML-N), and Pakistan Tehreek-e-Insaf (PTI) have played important roles in the country's political system.

Political Parties

Political parties are an important part of democracy. They represent the interests and opinions of different groups of people. In Pakistan, major political parties have different political programs and priorities.

The Pakistan Peoples Party (PPP) has historically focused on democracy, social welfare, and the interests of ordinary people. The Pakistan Muslim League-Nawaz (PML-N) has emphasized economic development, infrastructure, and business activity. The Pakistan Tehreek-e-Insaf (PTI) became a major political force by focusing on issues such as governance, accountability, and political reform.

Besides these parties, many regional and religious parties also participate in Pakistani politics.

Role of Democracy

Democracy gives citizens the right to choose their representatives through elections. Pakistan has held many general elections since its independence. Elections provide people with an opportunity to participate in the political process and hold governments accountable.

However, Pakistan's democratic system has faced several difficulties. Political conflicts, allegations of corruption, institutional tensions, election disputes, and changes in governments have sometimes weakened political stability.

Major Problems in Pakistani Politics

One of the biggest problems is political instability. Frequent political conflicts make it difficult for governments to focus on long-term policies.

Another important issue is corruption. Corruption can reduce public trust and negatively affect government institutions and economic development.

Political polarization is also a serious challenge. Supporters of different political parties often have strong disagreements, which can create tension in society.

Pakistan also faces challenges related to weak institutions, unemployment, inflation, poverty, and lack of effective governance. These problems cannot be solved only through political slogans; they require stable institutions, effective policies, and cooperation between political groups.

Role of the Military

The military has historically played an important role in Pakistan's political affairs. Pakistan has experienced periods of direct military rule as well as periods in which the military has had significant political influence.

Supporters of military involvement sometimes argue that the military can provide stability during political crises. Critics, however, argue that democratic institutions should be allowed to function independently and that political matters should be decided through constitutional and democratic processes.

A stable democracy requires clear constitutional boundaries and respect for civilian institutions.

Role of the Youth

Young people are an important part of Pakistan's population and have an important role in its political future. Young Pakistanis can contribute by voting responsibly, learning about political issues, avoiding misinformation, and participating peacefully in democratic activities.

Social media has also increased political awareness among young people. However, social media can spread false information and political hatred, so citizens should verify information before sharing it.

How Politics Can Improve

Pakistan can improve its political system by strengthening democratic institutions and ensuring transparent elections. Political parties should respect democratic values and accept peaceful disagreement.

The government should also focus on education, employment, economic development, and better public services. Strong institutions, an independent legal system, accountability, and respect for the Constitution can help create political stability.

Political leaders should put national interests above personal and party interests. At the same time, citizens should play their role by choosing representatives responsibly and respecting democratic processes.

Conclusion

Politics in Pakistan has experienced many successes and challenges since independence. The country has passed through democratic governments, military rule, political conflicts, and constitutional changes. Political instability, corruption, polarization, and weak governance remain important challenges.

The future of Pakistan depends on strong democratic institutions, responsible political leadership, an educated population, and active but peaceful participation by citizens. If political leaders and institutions work together according to the Constitution and prioritize the national interest, Pakistan can achieve greater political stability and economic development."""

In [ ]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
structured_model.invoke(prompt)

In [ ]:
class PPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float


In [ ]:
def evaluate_language(state: PPSCState):

    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [ ]:
def evaluate_analysis(state: PPSCState):

    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [ ]:
def evaluate_thought(state: PPSCState):

    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [ ]:
def final_evaluation(state: PPSCState):

    # summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content

    # avg calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}
    

In [ ]:
graph = StateGraph(PPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [ ]:
workflow

In [ ]:
intial_state = {
    'essay': essay
}

final_state=workflow.invoke(intial_state)

In [ ]:
print(final_state)